# Activation bottleneck on TinyStories — vast.ai

Four matched runs: a standard transformer, the same model with a **hard TopK
activation bottleneck** in front of every MLP, and that bottleneck trained with
the **adaptive LapSum Top-(K+J) surrogate gradient**.

This is the *activation*-sparsity experiment and is unrelated to the repo's
weight-sparsity methods (`ltp`, `cs`, `topk`); enabling both at once is a config
error. `W_in` and `W_out` here are ordinary dense layers trained normally.

**Instance setup**

* Template: any recent *PyTorch* image (CUDA + torch preinstalled).
* Disk: **≥ 60 GB** — token stream ~1 GB, HF cache a few GB, checkpoints ~1.3 GB
  each at 108M parameters.
* GPU: a single RTX 4090 / A5000 handles this comfortably.

Everything lives under `/workspace`, which survives instance stop/start.

In [ ]:
!nvidia-smi
!df -h /workspace | tail -1
!free -g | head -2
!nproc

## 1. The four setups

| run | config | forward through the MLP input | backward |
| --- | --- | --- | --- |
| **dense** | `configs/bn_dense.yaml` | the ordinary transformer | ordinary |
| **hard** | `configs/bn_hard.yaml` | `W_in` → hard TopK (K of N) → `W_out` | hard mask only; the J candidates get nothing |
| **lapsum** | `configs/bn_lapsum.yaml` | **identical to `hard`** | LapSum Top-(K+J) surrogate, adaptive bandwidth |
| **sched** | `configs/bn_sched.yaml` | **identical to `hard`** | same surrogate, temperature from a schedule instead of an `n_eff` solve |

The comparison is built to isolate one thing at a time:

* **dense vs hard** — what the activation bottleneck itself costs.
* **hard vs lapsum** — what the surrogate gradient buys. These two share an
  architecture, a parameter count and a *bit-identical forward pass*; they
  differ only in what flows backwards through the TopK boundary.
* **sched vs lapsum** — what the adaptive `n_eff` calibration buys over simply
  annealing the temperature. All four share the same forward pass.

Sizes: the dense baseline is **81.7M** parameters (10 layers, `d_model=640`,
GPT-Neo vocab). The bottleneck adds `2 · d_model · N` per bottlenecked layer —
**26.2M** at `N=2048` across all 10 layers, so the bottlenecked runs are 107.9M.
That gap is unavoidable (the projections are real parameters) and is why
`hard` — not `dense` — is the control for the surrogate.

## 2. Paths

In [ ]:
import os

WORKSPACE = '/workspace'                       # persistent volume mount point
REPO_DIR  = f'{WORKSPACE}/weight-sparsity'
DATA_DIR  = f'{WORKSPACE}/data/tinystories'    # tokenised token stream
RUNS_DIR  = f'{WORKSPACE}/runs_bottleneck'     # checkpoints + metrics
CACHE_DIR = f'{WORKSPACE}/hf_cache'            # keep the HF cache off the image layer

for p in (DATA_DIR, RUNS_DIR, CACHE_DIR):
    os.makedirs(p, exist_ok=True)

os.environ['HF_HOME'] = CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = f'{CACHE_DIR}/datasets'
os.environ['PYTHONUNBUFFERED'] = '1'

print('repo :', REPO_DIR)
print('data :', DATA_DIR)
print('runs :', RUNS_DIR)

## 3. Clone the repo and install

In [ ]:
REPO_URL = 'https://github.com/labofdoubt/weight-sparsity.git'

if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull -q

%cd $REPO_DIR
!git log --oneline -1

In [ ]:
# The PyTorch image already has torch -- install the package without it.
!pip install -q datasets transformers tokenizers pyyaml tqdm tensorboard
!pip install -q -e . --no-deps

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## 4. Prepare the data (once)

Downloads TinyStories, tokenises it with the GPT-Neo tokenizer and writes flat
`uint16` streams. All three setups read the same stream.

In [ ]:
TOKENIZER = 'gpt_neo'   # or 'bpe'
NUM_PROC  = 8           # tokenisation workers; keep <= nproc

cmd = (f'python -m wsparse.data --config configs/bn_dense.yaml '
       f'--data.data_dir={DATA_DIR} --data.tokenizer={TOKENIZER} '
       f'--data.tokenizer_path={DATA_DIR}/tokenizer --data.num_proc={NUM_PROC}')
!{cmd}

!ls -lh $DATA_DIR
!cat $DATA_DIR/meta.json

## 5. Pick a setup and the shared hyper-parameters

Run this notebook three times, once per `SETUP`, or set `SETUP` and re-run the
train cell. The overrides below apply to all three so the comparison stays fair;
the bottleneck knobs are ignored by `dense`.

In [ ]:
SETUP = 'lapsum'          # 'dense' | 'hard' | 'lapsum' | 'sched'

CONFIGS = {
    'dense':  'configs/bn_dense.yaml',
    'hard':   'configs/bn_hard.yaml',
    'lapsum': 'configs/bn_lapsum.yaml',
    'sched':  'configs/bn_sched.yaml',
}
RUN_NAMES = {name: f'bn_{name}' for name in CONFIGS}

CONFIG   = CONFIGS[SETUP]
RUN_NAME = RUN_NAMES[SETUP]

OVERRIDES = [
    f'--data.data_dir={DATA_DIR}',
    f'--train.out_dir={RUNS_DIR}',
    f'--train.run_name={RUN_NAME}',
    # --- shared across all three, so the comparison is fair ----------------
    '--train.max_steps=20000',
    '--train.batch_size=96',
    '--train.micro_batch_size=24',   # lower this if you hit OOM
    '--train.lr=6e-4',
    '--train.warmup_steps=500',
    '--train.dtype=bfloat16',
    '--train.compile=false',        # the gate graph-breaks; see the note below
    '--train.validate_every_steps=500',
    '--train.checkpoint_every_steps=2000',
    '--train.sample_every_steps=1000',
    '--train.seed=1337',            # same init for all three
    # --- bottleneck geometry (ignored by SETUP='dense') -------------------
    '--activation_bottleneck.n_features=2048',   # N
    '--activation_bottleneck.k=32',             # K active: 12.5% of N, 0.40x d_model
    '--activation_bottleneck.j=64',             # J extra candidates that only get gradient
    '--activation_bottleneck.n_eff=32',          # effective boundary participants
    '--activation_bottleneck.layers=all',        # all | even | odd | first:n | last:n | 0,2,4
    '--activation_bottleneck.selection_mode=abs_topk',   # topk | abs_topk
    # --- init calibration: match the block's output variance to its input ---
    # Uncalibrated, the K-sparse block attenuates the residual stream badly at
    # init (~0.19x for abs_topk, ~0.09x for gated_topk). This measures
    # std(in)/std(out) over a handful of batches and folds it into a frozen
    # scalar after the decoder. Costs a few forward passes, no training steps.
    '--activation_bottleneck.calibrate_output=true',
    '--activation_bottleneck.calibration_batches=4',   # batches per pass
    '--activation_bottleneck.calibration_iters=3',      # passes; layers are sequential,
                                                        # so the fix compounds down the stack
    # --- surrogate (the only thing that differs between hard and lapsum) ---
    '--activation_bottleneck.boundary_mode=both_sides',        # outside_only | both_sides
    '--activation_bottleneck.one_sided_weight_mode=true_gradient',  # score_softmax | true_gradient
    '--activation_bottleneck.effective_count_metric=ess',        # ess | entropy
    '--activation_bottleneck.surrogate_grad_scale=1.0',
    # Reweights only the J inactive candidates' gradient (1.0 = the exact VJP).
    # >1 pushes harder on exploration, 0 silences it entirely. Note this
    # deliberately breaks the per-row zero-sum property of the exact VJP, so the
    # surrogate can shift the whole barrier. Inert when surrogate_mode=hard.
    '--activation_bottleneck.inactive_grad_scale=1.0',
    # --- prescribed temperature (SETUP='sched' only; ignored by the others) --
    '--activation_bottleneck.temperature_schedule=linear',  # constant|linear|exponential|cosine|polynomial
    '--activation_bottleneck.temperature_start=1.0',   # broad boundary gradient early
    '--activation_bottleneck.temperature_end=0.01',    # sharp late
    '--activation_bottleneck.temperature_warmup_steps=0',
    '--activation_bottleneck.temperature_scale_mode=relative',   # relative | absolute
    '--activation_bottleneck.log_diagnostics=true',
]
OVERRIDE_STR = ' '.join(OVERRIDES)
print(SETUP, '->', CONFIG)
print(OVERRIDE_STR)

`train.compile=false`: the gate contains a custom autograd Function and
data-dependent solver control flow, so `torch.compile` graph-breaks around it.
The dense baseline can be compiled safely — but leave it off for all three if
you want the wall-clock numbers to be comparable.

### Parameter counts for all three

In [ ]:
for name, cfg in CONFIGS.items():
    print('=' * 78)
    !python scripts/model_summary.py --config $cfg $OVERRIDE_STR

## 6. Train

The log line adds `t` (adaptive temperature), `t/std` (relative to the score
scale), `neff` (realized effective count) and `dK` (`|Σp − K|`, the barrier
residual) for the bottlenecked runs.

**Watch `neff`**: if it drifts off its target, or `bottleneck/status_*` shows
rows flagged, the target is not attainable for that score geometry.

**In the foreground** (dies with the notebook connection):

In [ ]:
!python -m wsparse.train --config $CONFIG $OVERRIDE_STR

**Detached** (recommended — survives a disconnect). Run once, then poll the log.

In [ ]:
LOG = f'{RUNS_DIR}/{RUN_NAME}.log'
!mkdir -p $RUNS_DIR
!nohup python -m wsparse.train --config $CONFIG $OVERRIDE_STR > $LOG 2>&1 &
print('started, logging to', LOG)

In [ ]:
!tail -n 25 $LOG

Or queue all three back to back, detached — they share the data and the seed:

In [ ]:
ALL_LOG = f'{RUNS_DIR}/all_setups.log'
script = ' && '.join(
    f"python -m wsparse.train --config {cfg} "
    f"--data.data_dir={DATA_DIR} --train.out_dir={RUNS_DIR} "
    f"--train.run_name={RUN_NAMES[name]} "
    + ' '.join(o for o in OVERRIDES if not o.startswith(('--data.data_dir', '--train.out_dir', '--train.run_name')))
    for name, cfg in CONFIGS.items()
)
open('/tmp/run_all.sh', 'w').write(script + '\n')
!nohup bash /tmp/run_all.sh > $ALL_LOG 2>&1 &
print('queued dense -> hard -> lapsum, logging to', ALL_LOG)

In [ ]:
# stop everything
# !pkill -f 'wsparse.train'

## 7. TensorBoard

Every run writes events to `<RUNS_DIR>/<run_name>/tb`, so pointing TensorBoard
at `RUNS_DIR` overlays all four setups in one chart. Metric names are already
namespaced, so the panels group themselves:

| section | what to watch |
| --- | --- |
| `train/`, `val/` | `ce`, `ppl`, `loss`, `lr`, `grad_norm` |
| `perf/` | `tokens_per_s`, `ms_per_step` — the surrogate's real cost |
| `bottleneck/temperature*` | the adaptive bandwidth, absolute and relative to the score scale |
| `bottleneck/n_eff_*` | is the calibration hitting its target, and does the cheap shortcut differ (`n_eff_gap`) |
| `bottleneck/budget_residual` | `\|Σp − K\|`; should sit at ~1e-6, a rise means the barrier solve is struggling |
| `bottleneck/feature_*` | **dead features and usage collapse — see below** |
| `bottleneck/grad_*` | surrogate gradient magnitude on the active K, the exploratory J, and by candidate rank |
| `bottleneck/status_*`, `newton_failed` | solver fallbacks; should be 0 |

`bottleneck/feature_dead_frac` and `feature_usage_entropy` are the ones to keep
an eye on. The characteristic failure of a TopK bottleneck is collapse: a subset
of the N features wins every token, the rest are never selected, and their
`W_in`/`W_out` columns stop receiving gradient entirely — so the effective width
is far below N. The loss, the budget and `N_eff` all look perfectly healthy
while that happens. `feature_usage_entropy` is `exp(H)/N`: 1.0 is even usage,
and it falls towards the surviving fraction as features die.

In [ ]:
# vast.ai: forward the port from your machine with
#   ssh -N -L 6006:localhost:6006 -p <ssh_port> root@<ssh_host>
# then open http://localhost:6006
import subprocess, sys
subprocess.Popen([sys.executable, '-m', 'tensorboard.main',
                  '--logdir', RUNS_DIR, '--port', '6006', '--host', '0.0.0.0'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('tensorboard serving', RUNS_DIR, 'on port 6006')

# Or inline, if the notebook UI supports it:
# %load_ext tensorboard
# %tensorboard --logdir $RUNS_DIR --port 6006

## 8. Curves

In [ ]:
import json, os
import matplotlib.pyplot as plt

def load(name):
    path = f'{RUNS_DIR}/{RUN_NAMES[name]}/metrics.jsonl'
    if not os.path.exists(path):
        return []
    return [json.loads(l) for l in open(path)]

def series(records, key):
    xs = [(r['step'], r[key]) for r in records if key in r]
    return [x for x, _ in xs], [y for _, y in xs]

runs = {name: load(name) for name in CONFIGS}
runs = {k: v for k, v in runs.items() if v}
colors = {'dense': 'tab:grey', 'hard': 'tab:orange', 'lapsum': 'tab:blue',
          'sched': 'tab:green'}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

for name, rec in runs.items():
    s, v = series(rec, 'train/ce')
    axes[0].plot(s, v, color=colors[name], alpha=.55, lw=1)
    s, v = series(rec, 'val/ce')
    axes[0].plot(s, v, 'o-', color=colors[name], label=name, ms=3)
axes[0].set_xlabel('step'); axes[0].set_ylabel('cross-entropy')
axes[0].set_title('loss (faint = train, dots = val)'); axes[0].legend(); axes[0].grid(alpha=.3)

# what the surrogate is doing: adaptive bandwidth, relative to the score scale
for name, rec in runs.items():
    s, v = series(rec, 'bottleneck/temperature_rel')
    if s:
        axes[1].plot(s, v, color=colors[name], label=f'{name}: t / std(r)')
axes[1].set_xlabel('step'); axes[1].set_ylabel('t / std(scores)')
axes[1].set_title('adaptive bandwidth'); axes[1].legend(); axes[1].grid(alpha=.3)

# is the calibration hitting its target, and is the cheap shortcut exact?
for name, rec in runs.items():
    s, v = series(rec, 'bottleneck/n_eff_realized')
    if s:
        axes[2].plot(s, v, color=colors[name], label=f'{name}: realized')
    s, v = series(rec, 'bottleneck/n_eff_gap')
    if s:
        ax2 = axes[2].twinx()
        ax2.plot(s, v, ':', color=colors[name], alpha=.7)
        ax2.set_ylabel('n_eff_gap (true - score)')
axes[2].axhline(32, color='k', ls='--', lw=.8, label='target')
axes[2].set_xlabel('step'); axes[2].set_ylabel('N_eff')
axes[2].set_title('calibration (dotted = approximation gap)'); axes[2].legend(); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

## 9. Compare the setups

In [ ]:
import math

rows = []
for name, rec in runs.items():
    val = [r for r in rec if 'val/ce' in r]
    # filter on density, not temperature: the hard baseline runs no solver but
    # still reports the forward-side statistics
    last = [r for r in rec if 'bottleneck/density' in r]
    best = min((r['val/ce'] for r in val), default=float('nan'))
    row = {'setup': name,
           'best val ce': round(best, 4),
           'val ppl': round(math.exp(best), 2) if best == best else float('nan')}
    if last:
        d = last[-1]
        nan = float('nan')
        row.update({
            'K/N': round(d['bottleneck/density'], 4),
            't/std': round(d.get('bottleneck/temperature_rel', nan), 4),
            'N_eff': round(d.get('bottleneck/n_eff_realized', nan), 2),
            '|sum p - K|': f"{d['bottleneck/budget_residual']:.1e}" if 'bottleneck/budget_residual' in d else '-',
            'grad on J': f"{d.get('bottleneck/grad_inactive', 0.0):.2e}",
            'rows r_K+1 > b': round(d.get('bottleneck/frac_above_barrier', nan), 3),
        })
    rows.append(row)

try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index('setup'))
except ImportError:
    for r in rows:
        print(r)

print()
print('dense vs hard   : what the activation bottleneck itself costs.')
print('hard  vs lapsum : what the surrogate buys.  Same architecture, same')
print('                  parameter count, bit-identical forward pass -- the only')
print('                  difference is the backward.')
print()
print('grad on J is the tell: exactly 0.00e+00 for hard (the J inactive candidates')
print('receive nothing), non-zero for lapsum (they receive the boundary-exchange')
print('gradient).  NaNs in the t/std and N_eff columns are expected for dense and')
print('hard -- neither runs a temperature solve.')

## 10. Sample from a checkpoint

In [ ]:
CKPT = f'{RUNS_DIR}/{RUN_NAME}/latest.pt'
!python scripts/generate.py --ckpt "$CKPT" --prompt "Once upon a time" --tokens 200

## 11. Getting the results off the instance

`/workspace` survives a stop/start but **not** destruction. Copy what matters —
`metrics.jsonl` is small and holds every diagnostic.

In [ ]:
import shutil
dst = f'{WORKSPACE}/export'
os.makedirs(dst, exist_ok=True)
for name in RUN_NAMES.values():
    src = f'{RUNS_DIR}/{name}'
    if os.path.exists(src):
        os.makedirs(f'{dst}/{name}', exist_ok=True)
        for f in ('metrics.jsonl', 'config.yaml', 'summary.json'):
            if os.path.exists(f'{src}/{f}'):
                shutil.copy(f'{src}/{f}', f'{dst}/{name}/{f}')
print('exported metrics to', dst)
!du -sh $dst
!ls -R $dst | head -30

Knobs worth sweeping, in rough order of interest:

* `k` — the FLOP budget. `k=256` of `N=2048` is 12.5%, and 0.40× `d_model`.
* `j` — how many inactive candidates get gradient. `j=0` is not allowed; set
  `surrogate_mode=hard` for no surrogate at all.
* `n_eff` — how concentrated the boundary gradient is. Independent of `j`.
* `layers` — `even` halves the added parameters.
* `boundary_mode=both_sides` / `one_sided_weight_mode=true_gradient` — the exact
  calibrations, ~1.4–1.6× slower than the cheap default.